In [ ]:
%load_ext autoreload
%autoreload 2
# Import required libraries
import sys
sys.path.append("../")
import os
import numpy as np
import utm
import io
import shutil
import h5py
import json
import matplotlib.pyplot as plt
from PIL import Image
from typing import List
import cv2 
from pyproj import CRS, Transformer

# Load customized utility functions
from utils.transformations import rotation_matrix_from_angles, get_yaw_pitch_roll, filter_above_ground, compute_heading
from utils.plot import plot_lidar_camera_oxts
from utils.process import memory_usage

# Load ZOD DevKit
from zod import ZodFrames, ZodSequences, ZodDrives
import zod.constants as constants
from zod.constants import Camera, Lidar, Anonymization, AnnotationProject
from zod.data_classes.ego_motion import OXTS_TIMESTAMP_OFFSET, interpolate_transforms
from zod.data_classes import LidarData
from zod.utils.geometry import transform_points
from zod.visualization.lidar_on_image import get_3d_transform_camera_lidar
from zod.constants import Camera, Lidar
from zod.data_classes.metadata import SequenceMetadata

# Load aerial image download functions
from Map_Downloading.scripts.sweden import sweden_exact_position_image
from Map_Downloading.scripts.sweden import RESOLUTION as sweden_resolution
from Map_Downloading.scripts.france import france_exact_position_image
from Map_Downloading.scripts.france import TRUE_RESOLUTION as france_resolution


# Set matplotlib settings
%matplotlib inline

# NOTE! Set the path to dataset and choose a version
dataset_root = "/work/vita/datasets/zod"  # your local path to zod "/mnt/data/Work/datasets/zod"
version = "full"  # "mini" or "full"



# Load and display sensor frames
sensor_frames = plt.imread('./sensor_frames.png')
plt.figure(figsize=(20, 10))
plt.imshow(sensor_frames)
plt.axis('off')
plt.show()


In [ ]:
def approximate_meridian_convergence(lat_deg, lon_deg, central_meridian_deg):
    """
    Approximates the meridian convergence (in degrees) given a point's latitude and longitude,
    and the central meridian of the map projection.

    Args:
        lat_deg (float): Latitude in degrees.
        lon_deg (float): Longitude in degrees.
        central_meridian_deg (float): Central meridian longitude in degrees.

    Returns:
        float: Meridian convergence in degrees.
    """
    lat_rad = np.radians(lat_deg)
    lon_rad = np.radians(lon_deg)
    cm_rad = np.radians(central_meridian_deg)
    gamma_rad = np.arctan(np.tan(lon_rad - cm_rad) * np.sin(lat_rad))
    return np.degrees(gamma_rad)

In [ ]:
# initialize ZodDrives
zod_drives = ZodDrives(dataset_root=dataset_root, version=version)


# get default training and validation splits
training_drives = zod_drives.get_split(constants.TRAIN)
validation_drives = zod_drives.get_split(constants.VAL)

print(f"Number of training drives: {len(training_drives)}")
print(f"Number of validation drives: {len(validation_drives)}")

all_drive_num = list(training_drives) + list(validation_drives)
print(f"Number of total drives: {len(all_drive_num)}")

drives_per_country = {}

for i in range(len(all_drive_num)):
    zod_drive = zod_drives[all_drive_num[i]]
    # zod_drive._metadata = SequenceMetadata("/work/vita/datasets/zod/drives/000000/metadata.json")
    metadata = zod_drive.metadata
    
    if metadata.country_code in drives_per_country:
        drives_per_country[metadata.country_code].append(all_drive_num[i])
    else:
        drives_per_country[metadata.country_code] = [all_drive_num[i]]

for country in drives_per_country.keys():
    print(country, len(drives_per_country[country]))

In [ ]:
# rtk_per_frame = []

# for country, drive_indices in drives_per_country.items():
#     for drive_idx in drive_indices:
#         drive = zod_drives[drive_idx]
#         calibrations = drive.calibration
#         T_cam2oxts = calibrations.get_extrinsics(Camera.FRONT).transform
#         num_frames = len(drive.info.camera_frames['front_blur'])
#         oxts = drive.oxts

#         filename = f"{os.path.join(dataset_root, 'drives')}/{drive_idx}/oxts.hdf5"

#         with h5py.File(filename, "r") as f:
#             lat_oxts, lon_oxts, alt_oxts = f['posLat'][()][0], f['posLon'][()][0], f['posAlt'][()][0]
#             heading_oxts, pitch_oxts, roll_oxts = f['heading'][()][0], f['pitch'][()][0], f['roll'][()][0]
#             rtk_flag = f['isValid'][()]
#             timestamps = OXTS_TIMESTAMP_OFFSET + f['timestamp'][()] + f['leapSeconds'][()][0]

#         # Compute transformation from origin to UTM
#         T_origin2utm = np.eye(4)
#         r = rotation_matrix_from_angles(np.radians(roll_oxts), np.radians(pitch_oxts), np.radians(heading_oxts), order="ZYX")
#         T_origin2utm[:3, :3] = r

#         easting_oxts, northing_oxts, zone_number, zone_letter = utm.from_latlon(lat_oxts, lon_oxts)
#         T_origin2utm[0, 3] = northing_oxts
#         T_origin2utm[1, 3] = easting_oxts
#         T_origin2utm[2, 3] = alt_oxts

#         # Preallocate arrays
#         easting = np.zeros(num_frames)
#         northing = np.zeros(num_frames)
#         rtk_color_list = np.empty(num_frames, dtype=str)

#         for i, frame in enumerate(drive.info.camera_frames['front_blur']):
#             current_timestamp = frame.time.timestamp()
#             current_pose = oxts.get_poses(current_timestamp)

#             T_current2utm = T_origin2utm @ current_pose
#             easting[i] = T_current2utm[1, 3]
#             northing[i] = T_current2utm[0, 3]

#             closest_index = np.argmin(np.abs(timestamps - current_timestamp))
#             rtk_per_frame.append(rtk_flag[closest_index])
#             rtk_color_list[i] = 'g' if rtk_flag[closest_index] == 1 else 'r'

#         # Plot once per drive instead of multiple times
#         plt.figure(figsize=(8, 6))
#         plt.title(f"{country}, Drive ID: {drive_idx}")
#         plt.scatter(easting, northing, c=rtk_color_list, s=5, alpha=0.6)
#         plt.axis('equal')
#         plt.xlabel("Easting")
#         plt.ylabel("Northing")
#         plt.grid(True)
#         plt.show()

# print(f"RTK Valid Ratio: {np.mean(rtk_per_frame):.3f}")

### Use oxts of the first frame and ego-motion to get current pose (not recommend)

In [ ]:
# size = 100 # side length (m) of the requested aerial image


# # List of drives to process
# drive_list = ['000027']  # Modify as needed

# for drive_idx in drive_list:
#     drive = zod_drives[drive_idx]

#     # Define Paths
#     drive_path = os.path.join(dataset_root, 'drives', drive_idx)
#     aerial_path = os.path.join(drive_path, "aerial_img")
#     ground_path = os.path.join(drive_path, "ground_img")

#     # Create or Reset Directories
#     for path in [aerial_path, ground_path]:
#         if os.path.exists(path):
#             shutil.rmtree(path)
#             print(f"Removed existing folder: {path}")
#         os.makedirs(path)
#         print(f"Created new folder: {path}")

#     calibrations = drive.calibration

#     cam_intrinsics_4x3 = calibrations.cameras[Camera.FRONT].intrinsics
#     cam_intrinsics = cam_intrinsics_4x3[:, :3]
#     cam_distortion = calibrations.cameras[Camera.FRONT].distortion
#     image_dimensions = tuple(calibrations.cameras[Camera.FRONT].image_dimensions)

#     # === Compute undistortion maps ===
#     K_new = cv2.fisheye.estimateNewCameraMatrixForUndistortRectify(
#         cam_intrinsics, cam_distortion, image_dimensions, np.eye(3), balance=0.0
#     )
    
#     map1, map2 = cv2.fisheye.initUndistortRectifyMap(
#         cam_intrinsics, cam_distortion, np.eye(3), K_new, image_dimensions, cv2.CV_16SC2
#     )
    
#     T_cam2oxts = calibrations.get_extrinsics(Camera.FRONT).transform
#     T_lidar2oxts = calibrations.lidars[Lidar.VELODYNE].extrinsics.transform

#     num_frames = len(drive.info.camera_frames['front_blur'])
#     filename = os.path.join(drive_path, "oxts.hdf5")

#     # Read HDF5 Data Once
#     with h5py.File(filename, "r") as f:
#         lat_origin = f['posLat'][()][0]
#         lon_origin = f['posLon'][()][0]
#         alt_origin = f['posAlt'][()][0]
#         yaw_origin = 90 - f['heading'][()][0]
#         # if drive.metadata.country_code == 'SE':
#         #     yaw = yaw - 2.5  # Adjusted heading
#         pitch_origin = f['pitch'][()][0]
#         roll_origin = f['roll'][()][0]

#     T_origin2utm = np.eye(4)
#     T_origin2utm[:3, :3] = rotation_matrix_from_angles(
#         np.radians(roll_origin), np.radians(pitch_origin), np.radians(yaw_origin), order="ZYX"
#     )
    
#     easting_origin, northing_origin, zone_number, zone_letter = utm.from_latlon(lat_origin, lon_origin)
#     T_origin2utm[:3, 3] = [easting_origin, northing_origin, alt_origin]

#     heading_difference = []
#     T_prev = np.eye(4)

#     for i in range(0, num_frames):
#         if i % 100 == 0:
#             print(f"Frame {i}/{num_frames}, Memory Usage: {memory_usage():.2f} MB")

#         current_timestamp = drive.info.camera_frames['front_blur'][i].time.timestamp()
#         current_pose = drive.oxts.get_poses(current_timestamp)

#         T_current2utm = T_origin2utm @ current_pose

#         yaw_oxts, _, _ = get_yaw_pitch_roll(T_current2utm)
#         heading_oxts = 90 - np.degrees(yaw_oxts)

#         # Save Ground Image
#         grd_image = drive.info.camera_frames['front_blur'][i].read()
        
#         # === Apply remapping to undistort ===
#         grd_image = cv2.remap(grd_image, map1, map2, interpolation=cv2.INTER_LINEAR)

#         plt.imshow(grd_image)
#         plt.show()
#         # Image.fromarray(grd_image).resize((grd_image.shape[1] // 4, grd_image.shape[0] // 4), Image.LANCZOS).save(
#         #     os.path.join(ground_path, f'frame{i:06}.png'), format="PNG"
#         # )

#         # Process Lidar Data
#         pcd = drive.get_compensated_lidar(drive.info.camera_frames['front_blur'][i].time).points
#         pcd_utm = transform_points(pcd, T_current2utm @ T_lidar2oxts)
#         pcd_utm = filter_above_ground(pcd_utm, ground_threshold=1)

        
#         # Retrieve Aerial Image
#         lat_current, lon_current = utm.to_latlon(T_current2utm[0, 3], T_current2utm[1, 3], zone_number, zone_letter)
#         if drive.metadata.country_code == 'SE':
#             aerial_img = sweden_exact_position_image(lat_current, lon_current, size)
#             resolution = sweden_resolution
#         elif drive.metadata.country_code == 'FR':
#             aerial_img = france_exact_position_image(lat_current, lon_current, size)
#             resolution = france_resolution

#         # Plot BEV (Bird’s Eye View) with LiDAR points
#         bev_image = np.array(aerial_img)
#         H, W = bev_image.shape[:2]
#         bev_center = (T_current2utm[0, 3], T_current2utm[1, 3])

#         # Convert LiDAR points to BEV pixel coordinates
#         x_indices = ((pcd_utm[:, 0] - bev_center[0]) / resolution + W / 2).astype(int)
#         y_indices = H - ((pcd_utm[:, 1] - bev_center[1]) / resolution + H / 2).astype(int)
#         x_indices, y_indices = np.clip(x_indices, 0, W - 1), np.clip(y_indices, 0, H - 1)

#         plt.figure(figsize=(8, 8))
#         plt.imshow(bev_image, origin='upper')
#         plt.scatter(x_indices, y_indices, s=0.1, c='purple', alpha=0.3, label="LiDAR Points")
#         plt.quiver(W / 2, H / 2, np.sin(np.radians(heading_oxts)), np.cos(np.radians(heading_oxts)), color='r', scale=20, label='OXTS')
#         plt.legend(loc=2)
#         plt.axis('off')
#         # plt.savefig(os.path.join(aerial_path, f'frame{i:06}.png'), bbox_inches='tight')
#         plt.show()
#         plt.close()



### Use previous and next oxts measure (global positioning) to get current pose (recommended)

In [ ]:
size = 100 # side length (m) of the requested aerial image


# List of drives to process
drive_list = [str(i).zfill(6) for i in range(28)]

# drive_list = ['000021']  # Modify as needed

for drive_idx in drive_list:
    print('drive_idx', drive_idx)
    drive = zod_drives[drive_idx]

    # Define Paths
    drive_path = os.path.join(dataset_root, 'drives', drive_idx)
    aerial_path = os.path.join(drive_path, "aerial_img")
    ground_path = os.path.join(drive_path, "ground_img")

    # # Create or Reset Directories
    # for path in [aerial_path, ground_path]:
    #     if os.path.exists(path):
    #         shutil.rmtree(path)
    #         print(f"Removed existing folder: {path}")
    #     os.makedirs(path)
    #     print(f"Created new folder: {path}")

    calibrations = drive.calibration
    cam_intrinsics_4x3 = calibrations.cameras[Camera.FRONT].intrinsics
    cam_intrinsics = cam_intrinsics_4x3[:, :3]
    cam_distortion = calibrations.cameras[Camera.FRONT].distortion
    image_dimensions = tuple(calibrations.cameras[Camera.FRONT].image_dimensions)

    # === Compute undistortion maps ===
    K_new = cv2.fisheye.estimateNewCameraMatrixForUndistortRectify(
        cam_intrinsics, cam_distortion, image_dimensions, np.eye(3), balance=0.0
    )
    
    map1, map2 = cv2.fisheye.initUndistortRectifyMap(
        cam_intrinsics, cam_distortion, np.eye(3), K_new, image_dimensions, cv2.CV_16SC2
    )
    
    T_cam2oxts = calibrations.get_extrinsics(Camera.FRONT).transform
    T_lidar2oxts = calibrations.lidars[Lidar.VELODYNE].extrinsics.transform

    num_frames = len(drive.info.camera_frames['front_blur'])
    filename = os.path.join(drive_path, "oxts.hdf5")

    # Read HDF5 Data Once
    with h5py.File(filename, "r") as f:
        def recursively_print(name, obj):
            print(name)
        # f.visititems(recursively_print)
        # print('GNSSAntenna/accuracyHeadingAntennas', f['GNSSAntenna/accuracyHeadingAntennas'][()])
        # print('poseSource', f['poseSource'][()])

        orientationMode = f['orientationMode'][()]
        earth_frame_bytes = f['earthFrame'][()]
        earth_frame = b''.join(earth_frame_bytes).decode('utf-8') 

        INSToHostRotation = f['INSToHostRotation'][()]

        datumEllipsoid_bytes = f['datumEllipsoid'][()]
        datumEllipsoid = b''.join(datumEllipsoid_bytes).decode('utf-8') 
        
        lat = f['posLat'][()]
        lon = f['posLon'][()]
        alt = f['posAlt'][()]
        print(f['heading'][()] )
        yaw = 90 - f['heading'][()] 
        
        pitch = f['pitch'][()]
        roll = f['roll'][()]
        timestamp = OXTS_TIMESTAMP_OFFSET + f['timestamp'][()] + f['leapSeconds'][()][0]
    # print('earth_frame', earth_frame)
    # print('datumEllipsoid', datumEllipsoid)
    # print('INSToHostRotation', (INSToHostRotation))
    # print('orientationMode', orientationMode)
    
    heading_difference = []
    T_prev = np.eye(4)


    # for i in range(0, num_frames):
    for i in range(0, 1):
        if i % 100 == 0:
            print(f"Frame {i}/{num_frames}, Memory Usage: {memory_usage():.2f} MB")

        current_timestamp = drive.info.camera_frames['front_blur'][i].time.timestamp()
        
        # Use np.searchsorted for efficient timestamp lookup
        oxts_idx = np.searchsorted(timestamp, current_timestamp, side='right')

        if oxts_idx == 0 or oxts_idx >= len(timestamp):
            continue  # Skip if no valid index

        # Extract previous and next OXTS readings
        lat_prev, lon_prev, alt_prev = lat[oxts_idx - 1], lon[oxts_idx - 1], alt[oxts_idx - 1]
        yaw_prev, pitch_prev, roll_prev = yaw[oxts_idx - 1], pitch[oxts_idx - 1], roll[oxts_idx - 1]
        easting_prev, northing_prev, zone_number_prev, zone_letter_prev = utm.from_latlon(lat_prev, lon_prev)

        lat_next, lon_next, alt_next = lat[oxts_idx], lon[oxts_idx], alt[oxts_idx]
        yaw_next, pitch_next, roll_next = yaw[oxts_idx], pitch[oxts_idx], roll[oxts_idx]
        easting_next, northing_next, zone_number_next, zone_letter_next = utm.from_latlon(lat_next, lon_next)

        gamma_prev = 0
        gamma_next = 0
        if drive.metadata.country_code == 'SE':
            gamma_prev = approximate_meridian_convergence(lat_prev, lon_prev, 15)
            gamma_next = approximate_meridian_convergence(lat_next, lon_next, 15)
        
        yaw_prev += gamma_prev # heading should subtract gamma, hence yaw should be adding 
        yaw_next += gamma_next


        # Compute previous and next transformation matrices
        T_previous = np.eye(4)
        T_previous[:3, :3] = rotation_matrix_from_angles(
            np.radians(roll_prev), np.radians(pitch_prev), np.radians(yaw_prev), order="ZYX"
        )
        T_previous[:3, 3] = [easting_prev, northing_prev, alt_prev]

        T_next = np.eye(4)
        T_next[:3, :3] = rotation_matrix_from_angles(
            np.radians(roll_next), np.radians(pitch_next), np.radians(yaw_next), order="ZYX"
        )
        T_next[:3, 3] = [easting_next, northing_next, alt_next]

        # Interpolate transformation
        interp_factor = (current_timestamp - timestamp[oxts_idx - 1]) / (timestamp[oxts_idx] - timestamp[oxts_idx - 1])
        T_current = interpolate_transforms(T_previous, T_next, interp_factor)
        yaw_oxts, _, _ = get_yaw_pitch_roll(T_current)
        heading_oxts = 90 - np.degrees(yaw_oxts)

        # Save Ground Image
        grd_image = drive.info.camera_frames['front_blur'][i].read()
        # === Apply remapping to undistort ===
        grd_image = cv2.remap(grd_image, map1, map2, interpolation=cv2.INTER_LINEAR)
        
        Image.fromarray(grd_image).resize((grd_image.shape[1] // 4, grd_image.shape[0] // 4), Image.LANCZOS).save(
            os.path.join(ground_path, f'frame{i:06}.png'), format="PNG"
        )

        # Process Lidar Data
        pcd = drive.get_compensated_lidar(drive.info.camera_frames['front_blur'][i].time).points
        pcd_utm = transform_points(pcd, T_current @ T_lidar2oxts)
        pcd_utm = filter_above_ground(pcd_utm, ground_threshold=1)

        # Compute heading difference
        heading_move = compute_heading(T_prev[0, 3], T_prev[1, 3], T_current[0, 3], T_current[1, 3])

        if i > 0:
            difference = (heading_move - heading_oxts + 180) % 360 - 180
            # print('difference', difference)
            heading_difference.append(difference)

        T_prev = T_current.copy()

        # Retrieve Aerial Image
        lat_current, lon_current = utm.to_latlon(T_current[0, 3], T_current[1, 3], zone_number_next, zone_letter_next)
        if drive.metadata.country_code == 'SE':
            aerial_img = sweden_exact_position_image(lat_current, lon_current, size)
            resolution = sweden_resolution
        elif drive.metadata.country_code == 'FR':
            aerial_img = france_exact_position_image(lat_current, lon_current, size)
            resolution = france_resolution

        # Plot BEV (Bird’s Eye View) with LiDAR points
        bev_image = np.array(aerial_img)
        H, W = bev_image.shape[:2]
        bev_center = (T_current[0, 3], T_current[1, 3])

        # Convert LiDAR points to BEV pixel coordinates
        x_indices = ((pcd_utm[:, 0] - bev_center[0]) / resolution + W / 2).astype(int)
        y_indices = H - ((pcd_utm[:, 1] - bev_center[1]) / resolution + H / 2).astype(int)
        x_indices, y_indices = np.clip(x_indices, 0, W - 1), np.clip(y_indices, 0, H - 1)

        plt.figure(figsize=(8, 8))
        plt.imshow(bev_image, origin='upper')
        plt.scatter(x_indices, y_indices, s=0.1, c='purple', alpha=0.3, label="LiDAR Points")
        # plt.quiver(W / 2, H / 2, np.sin(np.radians(heading_move)), np.cos(np.radians(heading_move)), color='cyan', scale=20, label='Movement')
        plt.quiver(W / 2, H / 2, np.sin(np.radians(heading_oxts)), np.cos(np.radians(heading_oxts)), color='r', scale=20, label='OXTS')
        plt.legend(loc=2)
        plt.axis('off')
        # plt.savefig(os.path.join(aerial_path, f'frame{i:06}.png'), bbox_inches='tight')
        plt.show()
        plt.close()

    print(f"Drive {drive_idx} Completed. Heading Difference Mean: {np.mean(heading_difference):.2f}, Median: {np.median(heading_difference):.2f}")